In [26]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pprint import pprint
import gradio as gr
import json
import requests
import random

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print(pushover_user)
print(pushover_token)

if OPENAI_API_KEY is None:
    raise Exception ("API key is missing.")
else:
    print(OPENAI_API_KEY[:8])

client = OpenAI()

uj2zzdtmw5jhsyhrenptcbut58wgzs
a17pqtdb46zrxux3tughsdkykbqcgd
sk-proj-


### Step 2: Simple RAG w/guardrails & dynamic context injection

In [27]:
system_message = """
Use information only shared below. Do not go outside of the information shared.  To be specific use information only shared between the *** markers.
Any question asked that cannot be derived from the information shared between *** markers then explicitly call out i don't know.
***
You are a digitial twin of Dipesh Valia. I would like every one to address me by my first name Dipesh.
when replying you use Dipesh as a first name, using his voice, personality and knowledge.

About my background
I was born in Mumbai, India. Earlier Mumbai was known as Bombay. I completed my B.Tech frm Dr. B.A.Tech Univeristy in Lonere, Raigad in Petrochemical Engineering, an extension to Chemical Engineering.

I worked for 4 years before I came to US in 1997. I did my M.S in computer science from University of Houston and then MBA from Carnegie Mellon University from Teppere School of Business from 2006 to 2009

I worked at Aspect Communication AKA Aspect Software for 14 years. i grew from Software Engineering to Sr Manager in that tenure.
After Aspect, I worked at Walmartlabs. Worked in Cart and Checkout group and build next gen platform for the supply chain for the Sam's club.
Later I worked as a Director of Engineereing at MobileIron, which got acquired by Ivanti.
MobileIrion and Ivanti are in security space and managed Core Platform and Platform Services teams.

I enjoy doing hiking, biking and playing sports. In hiking i did Mt Whitney, Half dome (4 times) and Grand Canyon Rim-to-Rim. 
I completed 100 miles biking thrice.
I used to play Vollyball and now I'm playing badminton for last 3 years. I go twice a week to play badminton.

Besides sports i enjoy watching movies and spend time with my family.
I have a family of 4 (myself, wife and two kids 19 and 12) and a dog Ginger.

I like to learn new things and engage learning espcially in field of AI and doing hands on project. Currently I'm working on a Digital twin project.
 I attends seminars and conferences in field of AI.
 ***

"""

# Dynamic content but still rudimentary
topic_context = {
    "Born": "***I was born in Borivali, a suburb of Mumbai aka Bombay, Maharashtra, India in 1972***",
    "1976-1986" : "I did elementary, middle and higher secondary schooling in Mumbai, India",
    "1989-1993": "I was doing undergraudate at Dr.B.A.T.U in Petrochemical, a specialzed branch of Chemical Engineering. The University was located in Lonere, Raigad",
    "Hobbies2": "Hiking, biking, watching and playing Badminton",
    "Aspect Software": " Worked for 14 years from 2001 to 2014. Started as a Softward Engineer and grew upto Sr Manager. Aspect is mainly into Unified Communications Software",
    "1997-2000": " Masters in Computer Science at University of Houstong, Houston, Texas",
    "2006-2009" :" MBA from Carnegie Mellon Univeristy, Tepper School of Business"
}


### Step 3a Challenge to add a tool (pushover) to existing logic

In [28]:


def send_notification(message:str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)
    return message


send_notification_function = {

    "name" : "send_notification",
    "description": "Sends a push notification to the real Dipesh via pushover. Use this to alert the user about the change",
    "parameters": {
        "type": "object",
        "properties": {

            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required":["message"]
    }
}

tools = [{"type": "function", "function": send_notification_function}]




### Step 3b Add roll dice tool calling function

In [29]:
# add another notification - dice roll

#Simulates rolling  a single six-sided one
def dice_roll():
    result = random.randint(1,6)
    return result

#DESCRIBE FUNCTION FORM LLM
dice_roll_function = {

    "name" : "dice_roll",
    "description": "simulate rolling a dice to get a random number between 1 to 6. Use this when user wants to roll a dice and get a result",
    "parameters": {
        "type": "object",
        "properties": {},
        "required":[]
    }
}

# add function to teh list of tools of LLM
tools.append({"type": "function", "function": dice_roll_function})
print(tools)

[{'type': 'function', 'function': {'name': 'send_notification', 'description': 'Sends a push notification to the real Dipesh via pushover. Use this to alert the user about the change', 'parameters': {'type': 'object', 'properties': {'message': {'type': 'string', 'description': "The notification message to send to the user's device"}}, 'required': ['message']}}}, {'type': 'function', 'function': {'name': 'dice_roll', 'description': 'simulate rolling a dice to get a random number between 1 to 6. Use this when user wants to roll a dice and get a result', 'parameters': {'type': 'object', 'properties': {}, 'required': []}}}]


### Step 4 Handle mulitple tool calls

In [30]:
# handle tool call
def handle_tool_call(tool_calls):
    # .....
    # return what to add to our "coontext" about the tool call results, a dictionary

    tool_call_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name

        if(function_name == "send_notification"):
            args = json.loads(tool_call.function.arguments)
            #send notification
            result = send_notification(args["message"])
            content = f"Notification sent successfully: {result}"
        elif (function_name == "dice_roll"):
            content = f"Dice Roll: {dice_roll()}"
        else:
            content = f"Unknown function: {function_name}"

        print(content)
        tool_call_result = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "content": content
            }
        tool_call_results.append(tool_call_result)

    return tool_call_results


Step 5 - Response_OpenAI function called by Gradio

In [31]:
def response_openai(message, history):
    # dynamic injection
    system_message_dynamic = system_message;
    for key, value in topic_context.items():
        print(f"key: {key} message: {message}")
        if key.lower() in message.lower():
            print(f"key: {key} value: {value}")
            system_message_dynamic += "\n\n" + value

    
    messages = [{"role": "system", "content": system_message_dynamic}] + history + [{"role": "user", "content": message}, ]


    print(f"response_openai history: {history}")

    response = client.chat.completions.create(
        model= "gpt-4.1-mini",
        messages= messages,
        tools=tools
    )
    # reply = response.choices[0].message.content

    #check if model wants to call a tool
    message = response.choices[0].message
    print(f"response_openai message after the first LLM call: {message}")

    while message.tool_calls:
        
        from pprint import pprint
        pprint(message.tool_calls)
        
        tool_call_results = handle_tool_call(message.tool_calls) # send list of tool calls
        messages.append(message)
        # 'extend' is adding a list to an existing list as compared to 'append' you are adding an element to a list.
        # no need to loop each element to add to an existing list
        messages.extend(tool_call_results)
        print(f"response_openai before 2nd LLM call: {message}")
        response2 = client.chat.completions.create(
           messages = messages,
           model = "gpt-4.1-mini",
           tools = tools
        )
     
        message = response2.choices[0].message
        print(f"response_openai message after 2nd LLM call: {message}")
        # print("66666")
    
    yield message.content

gr.ChatInterface(fn=response_openai).queue().launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


key: Born message: hi there
key: 1976-1986 message: hi there
key: 1989-1993 message: hi there
key: Hobbies2 message: hi there
key: Aspect Software message: hi there
key: 1997-2000 message: hi there
key: 2006-2009 message: hi there
response_openai history: []
response_openai message after the first LLM call: ChatCompletionMessage(content="Hey Dipesh! How's it going? What can I do for you today?", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
key: Born message: roll dice twice for me
key: 1976-1986 message: roll dice twice for me
key: 1989-1993 message: roll dice twice for me
key: Hobbies2 message: roll dice twice for me
key: Aspect Software message: roll dice twice for me
key: 1997-2000 message: roll dice twice for me
key: 2006-2009 message: roll dice twice for me
response_openai history: [{'role': 'user', 'metadata': None, 'content': [{'text': 'hi there', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'cont